# ECG data preparation — C#

C# pipeline: single-channel WFDB format 16 at 250 Hz → resample to 200 Hz → fifth-order Butterworth high-pass (0.5 Hz) and notch (60 Hz, Q=35) → R peaks → one-to-one N/S/V/Q annotation matching (100 ms) → one-second windows → interpolation to 500 points and min–max normalization → fuzzy SQI → NPZ signals and CSV indexes.

The signals in `data/train`, `data/dev`, and `data/test` are already prepared. To repeat preparation, place raw `.hea/.dat/.atr` files under `data/raw/pXX/pXXXXX/` or set `ICENTIA_RAW_DIR`. Patient partitions come from `data/preparation_config.json`. New output is written to `data/reprepared/`.

Build with `dotnet build src/Aih.Ecg -c Release` from the repository root and use the **.NET (C#)** / Polyglot Notebooks kernel in this directory. Implementation: `src/Aih.Ecg/Preparation.cs`, `SignalProcessing.cs`, and `Wfdb.cs`.


In [ ]:
#r "../src/Aih.Ecg/bin/Release/net8.0/Aih.Ecg.dll"
using System;
using System.IO;
using System.Text.Json;
using Aih.Ecg;

var root = Dataset.FindRoot();
var data = Path.Combine(root, "data");
Console.WriteLine(data);


In [ ]:
var raw = Environment.GetEnvironmentVariable("ICENTIA_RAW_DIR") ?? Path.Combine(data, "raw");
var output = Path.Combine(data, "reprepared");
Console.WriteLine($"Input: {raw}\nOutput: {output}");


## Check the implementation

Tests cover resampling, filters, synthetic peak detection, annotation matching, normalization, SQI, and NPZ/WFDB input/output.


In [ ]:
SelfTest.Run();


## Full data preparation

Preparation is disabled when opening this notebook. Set `runPreparation = true` to prepare all recordings for the 20 clients. A limit of `1` recording per patient supports a smaller check; `null` processes all recordings. The output directory must be empty.

The pipeline exports client indexes, a manifest, signals, class counts, rejection statistics, and configuration. Written arrays are reread and validated. Parameters and representations are defined in C#; rounding can affect decisions close to thresholds.


In [ ]:
bool runPreparation = false;
int? limitRecordsPerPatient = null;
if (runPreparation)
{
    var prepared = Preparation.Run(data, raw, output, limitRecordsPerPatient);
    Console.WriteLine(JsonSerializer.Serialize(prepared, new JsonSerializerOptions { WriteIndented = true }));
}
else
{
    Console.WriteLine("Full preparation is disabled. Prepared data are available in data/.");
}


## Prepared-data counts

Q means unclassifiable and is not automatically an arrhythmia. SQI uses two detectors (smoothed gradient and Pan–Tompkins) and q/p/k/bas indexes in one-second windows; the detectors and window unit are an adaptation of fuzzy SQI.


In [ ]:
Console.WriteLine(File.ReadAllText(Path.Combine(data, "split_class_counts.csv")));
